# Policy32 Compact XLS-R 추론

25GB XLS-R 학습 checkpoint에서 추론에 불필요한 optimizer state를 제거하고, 32-class 발화 국적/언어권 분류를 실행한다.

```text
24.17 GiB 학습 checkpoint
  -> last_optimizer_state 제거
  -> 8.06 GiB 추론 checkpoint
  -> XLS-R feature + mean pooling
  -> Policy32 linear classifier
  -> Top-5와 추론 시간 출력
```

기존 측정 결과는 모델 크기 `24.17 GiB -> 8.06 GiB`, A100 로딩 시간 `41.42초 -> 34.05초`다. 실제 시간은 OS 디스크 캐시와 GPU 상태에 따라 달라질 수 있다.

## 1. 환경 및 경로

In [1]:
from pathlib import Path
import json
import random
import subprocess
import sys
import time

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Audio, display

PROJECT_ROOT = Path('$HOME')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PYTHON = Path('$HOME/miniconda3/envs/fairseq_xlsr/bin/python')
SOURCE_CKPT = PROJECT_ROOT / 'models/xlsr2_2B_1000k.pt'
COMPACT_CKPT = Path('$SPEECH_AI_DATA/models/xlsr2_2B_1000k_inference.pt')
CONVERTER = PROJECT_ROOT / 'scripts/create_xlsr_inference_checkpoint.py'
HEAD_CKPT = PROJECT_ROOT / 'taeho/policy32_othergpu_split_16s_probe/linear_best.pt'
TEST_CSV = PROJECT_ROOT / 'taeho/policy32_othergpu_split_seed42/test.csv'
FAIRSEQ_ROOT = PROJECT_ROOT / 'fairseq'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAMPLE_RATE = 16_000
MAX_SEC = 16.0

print('device:', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
print('source:', SOURCE_CKPT)
print('compact:', COMPACT_CKPT)

device: cuda
gpu: NVIDIA A100-SXM4-80GB
source: $SPEECH_AI_MODELS/xlsr2_2B_1000k.pt
compact: $SPEECH_AI_DATA/models/xlsr2_2B_1000k_inference.pt


## 2. 원본 checkpoint 구성 확인

`mmap=True`를 사용하므로 25GB 전체 tensor를 즉시 메모리에 복사하지 않고 checkpoint 구조와 tensor 용량을 확인한다.

In [2]:
def tensor_bytes(value, seen=None):
    if seen is None:
        seen = set()
    object_id = id(value)
    if object_id in seen:
        return 0
    seen.add(object_id)
    if torch.is_tensor(value):
        return value.numel() * value.element_size()
    if isinstance(value, dict):
        return sum(tensor_bytes(item, seen) for item in value.values())
    if isinstance(value, (list, tuple, set)):
        return sum(tensor_bytes(item, seen) for item in value)
    return 0

source_state = torch.load(
    str(SOURCE_CKPT), map_location='cpu', mmap=True, weights_only=False
)
component_table = pd.DataFrame([
    {'component': key, 'tensor_GiB': tensor_bytes(value) / 1024**3}
    for key, value in source_state.items()
]).sort_values('tensor_GiB', ascending=False)
display(component_table.style.format({'tensor_GiB': '{:.3f}'}))
del source_state

,component,tensor_GiB
7,last_optimizer_state,16.115
2,model,8.058
6,extra_state,0.000
0,args,0.000
1,cfg,0.000
3,criterion,0.000
4,optimizer_history,0.000
5,task_state,0.000


## 3. 추론 전용 checkpoint 생성

Fairseq 호환에 필요한 작은 메타데이터는 유지하고, 약 16GB인 `last_optimizer_state`만 제거한다. 이미 compact checkpoint가 있으면 재생성하지 않는다.

In [3]:
REBUILD_COMPACT = False

if COMPACT_CKPT.exists() and not REBUILD_COMPACT:
    print('compact checkpoint already exists; conversion skipped')
else:
    command = [
        str(PYTHON), str(CONVERTER),
        '--input', str(SOURCE_CKPT),
        '--output', str(COMPACT_CKPT),
    ]
    if REBUILD_COMPACT:
        command.append('--force')
    completed = subprocess.run(command, check=True, text=True, capture_output=True)
    print(completed.stdout)

assert COMPACT_CKPT.is_file(), COMPACT_CKPT

compact checkpoint already exists; conversion skipped


In [4]:
source_gib = SOURCE_CKPT.stat().st_size / 1024**3
compact_gib = COMPACT_CKPT.stat().st_size / 1024**3
size_table = pd.DataFrame([
    {'checkpoint': 'original training', 'size_GiB': source_gib},
    {'checkpoint': 'compact inference', 'size_GiB': compact_gib},
])
display(size_table.style.format({'size_GiB': '{:.3f}'}))
print(f'reduction: {(1 - compact_gib/source_gib)*100:.2f}%')

,checkpoint,size_GiB
0,original training,24.173
1,compact inference,8.058


reduction: 66.67%


## 4. Compact XLS-R와 Policy32 linear head 로딩

모델 로딩 시간에는 checkpoint 읽기, Fairseq 모델 생성, GPU 전송이 포함된다.

In [5]:
from country_classifier.audio import load_wav_mono
from country_classifier.model import build_model

def sync_cuda():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)

head_payload = torch.load(HEAD_CKPT, map_location='cpu')
labels = list(head_payload['labels'])
classifier = nn.Linear(1920, len(labels))
classifier.load_state_dict(head_payload['head'])
classifier = classifier.to(DEVICE).eval()
x_mean = torch.as_tensor(head_payload['mu'], dtype=torch.float32, device=DEVICE)
x_std = torch.as_tensor(head_payload['sd'], dtype=torch.float32, device=DEVICE)

load_started = time.perf_counter()
xlsr_model = build_model(
    backend='fairseq_xlsr',
    num_classes=len(labels),
    checkpoint_path=COMPACT_CKPT,
    fairseq_root=FAIRSEQ_ROOT,
    freeze_encoder=True,
).to(DEVICE).eval()
sync_cuda()
model_load_sec = time.perf_counter() - load_started

print(f'model loaded: {model_load_sec:.2f} sec')
print('classes:', len(labels))
print('last class:', labels[-1])

INFO:fairseq.tasks.text_to_speech:Please install tensorboardX: pip install tensorboardX
$HOME/miniconda3/envs/fairseq_xlsr/lib/python3.8/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


model loaded: 34.72 sec
classes: 32
last class: 한국어_모국어


## 5. 추론 함수

In [6]:
def predict_audio(audio_path, top_k=5, max_sec=MAX_SEC):
    audio_path = Path(audio_path)
    total_started = time.perf_counter()

    waveform, _ = load_wav_mono(audio_path, target_sample_rate=SAMPLE_RATE)
    original_sec = waveform.numel() / SAMPLE_RATE
    waveform = waveform[:int(max_sec * SAMPLE_RATE)].contiguous()
    used_sec = waveform.numel() / SAMPLE_RATE
    waveforms = waveform.unsqueeze(0).to(DEVICE)
    padding_mask = torch.zeros_like(waveforms, dtype=torch.bool)

    sync_cuda()
    encoder_started = time.perf_counter()
    with torch.inference_mode():
        if DEVICE.type == 'cuda':
            with torch.autocast(device_type='cuda'):
                features, feature_mask = xlsr_model._extract_features(waveforms, padding_mask)
                pooled = xlsr_model._pool(features, feature_mask)
        else:
            features, feature_mask = xlsr_model._extract_features(waveforms, padding_mask)
            pooled = xlsr_model._pool(features, feature_mask)
    sync_cuda()
    encoder_sec = time.perf_counter() - encoder_started

    classifier_started = time.perf_counter()
    with torch.inference_mode():
        logits = classifier((pooled.float() - x_mean) / x_std)
        probabilities = torch.softmax(logits, dim=-1)[0]
        scores, indices = probabilities.topk(min(top_k, len(labels)))
    sync_cuda()
    classifier_sec = time.perf_counter() - classifier_started
    total_sec = time.perf_counter() - total_started

    predictions = pd.DataFrame([
        {
            'rank': rank,
            'label_id': int(index),
            'label': labels[int(index)],
            'probability_percent': float(score) * 100,
        }
        for rank, (score, index) in enumerate(zip(scores.cpu(), indices.cpu()), start=1)
    ])
    timing = {
        'model_load_sec': model_load_sec,
        'encoder_sec': encoder_sec,
        'classifier_sec': classifier_sec,
        'inference_total_sec': total_sec,
        'realtime_factor': total_sec / max(used_sec, 1e-9),
    }
    return predictions, timing, original_sec, used_sec

## 6. Test에서 랜덤 음성 추론

In [7]:
SAMPLE_GROUP = 'all'  # 'all', 'foreign', 'korean'
RANDOM_SEED = None

test_df = pd.read_csv(TEST_CSV)
if SAMPLE_GROUP == 'foreign':
    candidates = test_df[test_df['label_id'] != 31]
elif SAMPLE_GROUP == 'korean':
    candidates = test_df[test_df['label_id'] == 31]
else:
    candidates = test_df

random_state = RANDOM_SEED
if random_state is None:
    random_state = random.SystemRandom().randint(0, 2**32 - 1)
sample = candidates.sample(n=1, random_state=random_state).iloc[0]
audio_path = Path(sample['audio_path'])

print('true label:', sample['label_name'])
print('speaker:', sample['speaker_id'])
print('audio:', audio_path)
display(Audio(filename=str(audio_path)))

true label: 희소_슬라브어권
speaker: EX0239
audio: $SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/3. 한국생활II/EX37RA165_EX0239_20211108.wav


In [8]:
predictions, timing, original_sec, used_sec = predict_audio(audio_path)
top5_labels = predictions['label'].tolist()

print('true label:', sample['label_name'])
print('predicted:', predictions.iloc[0]['label'])
print('top1 correct:', predictions.iloc[0]['label'] == sample['label_name'])
print('top5 correct:', sample['label_name'] in top5_labels)
print(f'audio: {original_sec:.3f} sec, used: {used_sec:.3f} sec')
display(predictions.style.format({'probability_percent': '{:.4f}%'}))
display(pd.DataFrame([timing]).T.rename(columns={0: 'seconds'}).style.format('{:.6f}'))

true label: 희소_슬라브어권
predicted: 러시아어
top1 correct: False
top5 correct: False
audio: 11.450 sec, used: 11.450 sec


$HOME/miniconda3/envs/fairseq_xlsr/lib/python3.8/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


,rank,label_id,label,probability_percent
0,1,3,러시아어,88.4264%
1,2,14,일본어,7.9802%
2,3,16,중국어권,1.1096%
3,4,5,몽골어,0.7663%
4,5,23,희소_게르만어권,0.4595%


,seconds
model_load_sec,34.723609
encoder_sec,0.271945
classifier_sec,0.027786
inference_total_sec,0.304702
realtime_factor,0.026611


## 7. Test 클래스별 1개 샘플 추론

Test에 실제 존재하는 각 클래스에서 음성 1개를 선택해 추론한다. 공정한 시간 비교를 위해 첫 음성으로 1회 warm-up을 수행하고 집계에서 제외한다. Test에 없는 출력 클래스는 별도 표로 표시한다.

In [9]:
CLASS_SAMPLE_SEED = 42
RUN_WARMUP = True

test_label_ids = set(test_df['label_id'].astype(int).unique())
missing_test_classes = pd.DataFrame([
    {'label_id': label_id, 'class': label}
    for label_id, label in enumerate(labels)
    if label_id not in test_label_ids
])

class_samples = (
    test_df.sort_values(['label_id', 'audio_path'])
    .groupby('label_id', group_keys=False)
    .apply(lambda group: group.sample(n=1, random_state=CLASS_SAMPLE_SEED + int(group.name)))
    .sort_values('label_id')
    .reset_index(drop=True)
)

if RUN_WARMUP and len(class_samples):
    _ = predict_audio(class_samples.iloc[0]['audio_path'])
    print('warm-up completed; excluded from timing table')

classwise_results = []
for order, row in class_samples.iterrows():
    output, sample_timing, original_sec, used_sec = predict_audio(row['audio_path'])
    true_label = str(row['label_name'])
    predicted_label = str(output.iloc[0]['label'])
    top5_labels = output['label'].tolist()
    classwise_results.append({
        'label_id': int(row['label_id']),
        'class': true_label,
        'test_samples': int((test_df['label_id'] == row['label_id']).sum()),
        'speaker_id': row['speaker_id'],
        'predicted': predicted_label,
        'top1_probability_%': float(output.iloc[0]['probability_percent']),
        'top1_correct': predicted_label == true_label,
        'top5_correct': true_label in top5_labels,
        'audio_sec': original_sec,
        'used_sec': used_sec,
        'encoder_sec': sample_timing['encoder_sec'],
        'classifier_sec': sample_timing['classifier_sec'],
        'inference_total_sec': sample_timing['inference_total_sec'],
        'realtime_factor': sample_timing['realtime_factor'],
        'audio_path': row['audio_path'],
    })
    print(f"[{order + 1:02d}/{len(class_samples)}] {true_label}: "
          f"{predicted_label}, {sample_timing['inference_total_sec']:.4f}s")

classwise_df = pd.DataFrame(classwise_results)
display(classwise_df.style.format({
    'top1_probability_%': '{:.3f}',
    'audio_sec': '{:.3f}',
    'used_sec': '{:.3f}',
    'encoder_sec': '{:.4f}',
    'classifier_sec': '{:.6f}',
    'inference_total_sec': '{:.4f}',
    'realtime_factor': '{:.4f}',
}))

print('Classes absent from test:')
display(missing_test_classes)

warm-up completed; excluded from timing table
[01/30] 국가명_불명확: 태국어, 0.0755s
[02/30] 네팔어: 영어, 0.0791s
[03/30] 독일어: 영어, 0.0713s
[04/30] 러시아어: 몽골어, 0.0719s
[05/30] 말레이어: 네팔어, 0.0690s
[06/30] 몽골어: 몽골어, 0.0775s
[07/30] 미얀마어: 베트남어, 0.0711s
[08/30] 베트남어: 베트남어, 0.0757s
[09/30] 스페인어: 중국어권, 0.0739s
[10/30] 아랍어: 영어, 0.0766s
[11/30] 암하라어: 러시아어, 0.0689s
[12/30] 영어: 중국어권, 0.0720s
[13/30] 우즈베크어: 몽골어, 0.0678s
[14/30] 인도네시아어: 중국어권, 0.0714s
[15/30] 일본어: 일본어, 0.0724s
[16/30] 중국어권: 중국어권, 0.0721s
[17/30] 크메르어: 태국어, 0.0721s
[18/30] 타갈로그어: 일본어, 0.0723s
[19/30] 태국어: 베트남어, 0.0718s
[20/30] 터키어: 몽골어, 0.0725s
[21/30] 프랑스어: 영어, 0.0700s
[22/30] 희소_게르만어권: 인도네시아어, 0.0690s
[23/30] 희소_니제르콩고어권: 영어, 0.0717s
[24/30] 희소_로망스어권: 스페인어, 0.0718s
[25/30] 희소_슬라브어권: 러시아어, 0.0705s
[26/30] 희소_이란어권: 중국어권, 0.0693s
[27/30] 희소_인도아리아어권: 영어, 0.0700s
[28/30] 희소_인도유럽_기타: 영어, 0.0750s
[29/30] 희소_튀르크어권: 영어, 0.0723s
[30/30] 한국어_모국어: 한국어_모국어, 0.0821s


,label_id,class,test_samples,speaker_id,predicted,top1_probability_%,top1_correct,top5_correct,audio_sec,used_sec,encoder_sec,classifier_sec,inference_total_sec,realtime_factor,audio_path
0,0,국가명_불명확,12,EX0182,태국어,97.996,False,False,18.120,16.000,0.0744,0.000218,0.0755,0.0047,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/2.Validation/원천데이터/VS_6. 기타/2. 한국생활I/EX21RB009_EX0182_20210731.wav
1,1,네팔어,9,EX0057,영어,45.433,False,False,12.755,12.755,0.0727,0.000215,0.0791,0.0062,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/2. 한국생활I/EX20QB254_EX0057_20210807.wav
2,2,독일어,36,EX0538,영어,46.665,False,False,18.360,16.000,0.0680,0.000213,0.0713,0.0045,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/2. 한국생활I/EX29RB214_EX0538_20211006.wav
3,3,러시아어,1682,EX1012,몽골어,99.645,False,True,13.576,13.576,0.0709,0.000221,0.0719,0.0053,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/4. 한국문화I/EX43RB058_EX1012_20211115.wav
4,4,말레이어,1378,EX0025,네팔어,34.127,False,False,19.432,16.000,0.0680,0.000220,0.0690,0.0043,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/4. 한국문화I/EX40QB264_EX0025_20210817.wav
5,5,몽골어,5755,EX0841,몽골어,81.009,True,True,11.038,11.038,0.0716,0.000223,0.0775,0.0070,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/5. 한국문화II/EX57RA159_EX0841_20211029.wav
6,6,미얀마어,4403,EX0467,베트남어,68.834,False,True,16.440,16.000,0.0682,0.000217,0.0711,0.0044,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/3. 한국생활II/EX30QA275_EX0467_20210929.wav
7,7,베트남어,13570,VN0075,베트남어,89.171,True,True,13.296,13.296,0.0709,0.000219,0.0757,0.0057,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/TS_1. 베트남어/3. 한국생활II/VN31RA008_VN0075_20211124.wav
8,8,스페인어,1398,EX0705,중국어권,27.327,False,True,12.236,12.236,0.0730,0.000217,0.0739,0.0060,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/3. 한국생활II/EX35RA114_EX0705_20211006.wav
9,9,아랍어,146,EX0820,영어,79.412,False,False,14.174,14.174,0.0717,0.000228,0.0766,0.0054,$SPEECH_AI_DATA/aihub/foreign_korean_speech/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터_230113_add/TS_6. 기타_add/1. 한국일반/EX15RC114_EX0820_20211014.wav


Classes absent from test:


,label_id,class
0,15,종카어
1,19,타밀어


In [10]:
classwise_summary = pd.DataFrame({
    'metric': [
        'model_load_sec',
        'classes_in_test',
        'classes_absent_from_test',
        'sample_top1_accuracy',
        'sample_top5_accuracy',
        'mean_encoder_sec',
        'mean_classifier_sec',
        'mean_inference_total_sec',
        'median_inference_total_sec',
        'p95_inference_total_sec',
        'mean_realtime_factor',
    ],
    'value': [
        model_load_sec,
        len(classwise_df),
        len(missing_test_classes),
        classwise_df['top1_correct'].mean(),
        classwise_df['top5_correct'].mean(),
        classwise_df['encoder_sec'].mean(),
        classwise_df['classifier_sec'].mean(),
        classwise_df['inference_total_sec'].mean(),
        classwise_df['inference_total_sec'].median(),
        classwise_df['inference_total_sec'].quantile(0.95),
        classwise_df['realtime_factor'].mean(),
    ],
})
display(classwise_summary.style.format({'value': '{:.6f}'}))

,metric,value
0,model_load_sec,34.723609
1,classes_in_test,30.000000
2,classes_absent_from_test,2.000000
3,sample_top1_accuracy,0.166667
4,sample_top5_accuracy,0.500000
5,mean_encoder_sec,0.069894
6,mean_classifier_sec,0.000251
7,mean_inference_total_sec,0.072553
8,median_inference_total_sec,0.071946
9,p95_inference_total_sec,0.078347


## 8. 임의 WAV 파일

웹 녹음 또는 별도 WAV를 테스트할 때 아래 경로를 변경한다.

In [ ]:
# CUSTOM_AUDIO = Path('/path/to/audio.wav')
# custom_predictions, custom_timing, _, _ = predict_audio(CUSTOM_AUDIO)
# display(Audio(filename=str(CUSTOM_AUDIO)))
# display(custom_predictions.style.format({'probability_percent': '{:.4f}%'}))
# display(pd.DataFrame([custom_timing]).T.rename(columns={0: 'seconds'}))

## 운영 시 주의사항

- Compact checkpoint는 용량과 cold-start를 줄이지만 XLS-R 2B 모델 생성과 GPU 전송 비용은 남는다.
- 웹서비스에서는 요청마다 모델을 로드하지 않고 서버 시작 시 한 번만 로드해야 한다.
- `tensorboardX`와 `weight_norm deprecated` 메시지는 추론 실패를 의미하는 오류가 아니다.
- 이 모델은 법적 국적을 판정하지 않는다. 발화에서 관찰되는 모국어/언어권 특성을 분류하는 실험 모델이다.